<h1 style='color:green'>Naive Bayes Algorithm</h1>

<h2 style='color:blue'>Before drive to code, we understand some basic theory</h2>

[View the PDF](./011-Naive-Bayes-Algorithm.pdf)

In [4]:
from IPython.display import IFrame
IFrame("011-Naive-Bayes-Algorithm.pdf", width=1000, height=600)

### Naive Bayes Code - Scratch

In [6]:
import numpy as np
import pandas as pd 

In [7]:
golf = pd.read_csv("golf.csv")

In [8]:
golf

,Outlook,Temperature,Humidity,Windy,Play
0,sunny,hot,high,False,no
1,sunny,hot,high,True,no
2,overcast,hot,high,False,yes
3,rainy,mild,high,False,yes
4,rainy,cool,normal,False,yes
5,rainy,cool,normal,True,no
6,overcast,cool,normal,True,yes
7,sunny,mild,high,False,no
8,sunny,cool,normal,False,yes
9,rainy,mild,normal,False,yes


In [9]:
def prior_prob(golf, label):
    total_examples = golf.shape[0]
    class_example = (golf['Play'] == label).sum()
    return class_example/total_examples

In [10]:
PRIOR = {
    'yes' : prior_prob(golf, 'yes'),
    'no' : prior_prob(golf, 'no')
}

print(PRIOR)

{'yes': 0.6428571428571429, 'no': 0.35714285714285715}


In [11]:
def cond_prob(golf, feature, feature_value, label): 
    filtered_data = golf[golf['Play'] == label] 
    numerator = (filtered_data[feature] == feature_value).sum()
    denominator = filtered_data.shape[0]
    return numerator/denominator

In [12]:
cond_prob(golf, 'Windy', False, 'no')

0.4

In [13]:
features = list(golf.columns)[:-1]
COND_PROB = {}

for label in golf['Play'].unique(): 
    COND_PROB[label]={} 
    for feature in features: 
        COND_PROB[label][feature]={}
        feature_values = golf[feature].unique()
        for fea_val in feature_values:
            #no, Outlook, sunny
            prob = round(cond_prob(golf, feature, fea_val, label),2)
            COND_PROB[label][feature][fea_val]=prob
            print(label, feature, fea_val, prob)

no Outlook sunny 0.6
no Outlook overcast 0.0
no Outlook rainy 0.4
no Temperature hot 0.4
no Temperature mild 0.4
no Temperature cool 0.2
no Humidity high 0.8
no Humidity normal 0.2
no Windy False 0.4
no Windy True 0.6
yes Outlook sunny 0.22
yes Outlook overcast 0.44
yes Outlook rainy 0.33
yes Temperature hot 0.22
yes Temperature mild 0.44
yes Temperature cool 0.33
yes Humidity high 0.33
yes Humidity normal 0.67
yes Windy False 0.67
yes Windy True 0.33


In [14]:
COND_PROB

{'no': {'Outlook': {'sunny': 0.6, 'overcast': 0.0, 'rainy': 0.4},
  'Temperature': {'hot': 0.4, 'mild': 0.4, 'cool': 0.2},
  'Humidity': {'high': 0.8, 'normal': 0.2},
  'Windy': {False: 0.4, True: 0.6}},
 'yes': {'Outlook': {'sunny': 0.22, 'overcast': 0.44, 'rainy': 0.33},
  'Temperature': {'hot': 0.22, 'mild': 0.44, 'cool': 0.33},
  'Humidity': {'high': 0.33, 'normal': 0.67},
  'Windy': {False: 0.67, True: 0.33}}}

### Prediction

In [16]:
X_test = ["sunny", "hot", "normal", False]

In [17]:
for label in golf['Play'].unique(): 
    prior = PRIOR[label]
    likelihood = 1.0

    for i in range(len(features)): 
        feature = features[i]
        fea_value = X_test[i] 

        likelihood *= COND_PROB[label][feature][fea_value]

    post = likelihood * prior

    print(label, post)

no 0.006857142857142858
yes 0.013967202857142858


In [96]:
0.006/(0.006+0.013)


0.31578947368421056

In [98]:
0.01396/(0.006+0.013)

0.7347368421052631

# Implementation - Naive Bayes Sklearn

In [19]:
golf = pd.read_csv('golf.csv')

In [20]:
golf

,Outlook,Temperature,Humidity,Windy,Play
0,sunny,hot,high,False,no
1,sunny,hot,high,True,no
2,overcast,hot,high,False,yes
3,rainy,mild,high,False,yes
4,rainy,cool,normal,False,yes
5,rainy,cool,normal,True,no
6,overcast,cool,normal,True,yes
7,sunny,mild,high,False,no
8,sunny,cool,normal,False,yes
9,rainy,mild,normal,False,yes


In [21]:
from sklearn.preprocessing import LabelEncoder

In [22]:
le1 = LabelEncoder()
golf['Outlook'] = le1.fit_transform(golf['Outlook'])

In [23]:
golf

,Outlook,Temperature,Humidity,Windy,Play
0,2,hot,high,False,no
1,2,hot,high,True,no
2,0,hot,high,False,yes
3,1,mild,high,False,yes
4,1,cool,normal,False,yes
5,1,cool,normal,True,no
6,0,cool,normal,True,yes
7,2,mild,high,False,no
8,2,cool,normal,False,yes
9,1,mild,normal,False,yes


In [24]:
le2 = LabelEncoder()
golf['Temperature'] = le2.fit_transform(golf['Temperature'])

le3 = LabelEncoder()
golf['Humidity'] = le3.fit_transform(golf['Humidity'])

le4 = LabelEncoder()
golf['Windy'] = le4.fit_transform(golf['Windy'])

le5 = LabelEncoder()
golf['Play'] = le5.fit_transform(golf['Play'])


In [25]:
golf

,Outlook,Temperature,Humidity,Windy,Play
0,2,1,0,0,0
1,2,1,0,1,0
2,0,1,0,0,1
3,1,2,0,0,1
4,1,0,1,0,1
5,1,0,1,1,0
6,0,0,1,1,1
7,2,2,0,0,0
8,2,0,1,0,1
9,1,2,1,0,1


In [26]:
X = golf.iloc[:,:-1]
y = golf.iloc[:,-1]

In [27]:
from sklearn.naive_bayes import CategoricalNB

In [28]:
model = CategoricalNB()

In [29]:
model.fit(X,y)

CategoricalNB()

### Prediction

In [31]:
X_test = ["sunny", "hot", "normal", False]

In [80]:
X_test = [
    le1.transform(['sunny'])[0],
    le2.transform(['hot'])[0],
    le3.transform(['normal'])[0],
    le4.transform([False])[0]
]
X_test

[2, 1, 1, 0]

In [33]:
model.predict([X_test])

C:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but CategoricalNB was fitted with feature names
  warnings.warn(


array([1])

##### Warning Because: Passing a plain list or NumPy array to model.predict(...), but the model was trained using a Pandas DataFrame with column names (golf).

In [88]:
import pandas as pd

X_test_named = pd.DataFrame([X_test], columns=['Outlook', 'Temperature', 'Humidity', 'Windy'])
X_test_named

,Outlook,Temperature,Humidity,Windy
0,2,1,1,0


In [90]:
model.predict(X_test_named)

array([1])

In [92]:
model.predict_proba(X_test_named)

array([[0.33508723, 0.66491277]])